# Theil-T Decomposition and Shapley Decomposition of Theil-T

Decomposes inequality in the 2023 consumption-based biodiversity footprint

## Part 1 — Multiplier vectors and expenditure/basket-share inputs

In [5]:
import gc
import os
import time

import h5py
import numpy as np
import pandas as pd
import scipy.io

# ── Paths (adjust to your local data directory) ────────────────────────────
IO_DIR  = r'E:\phdstudy\IO'
HOU_DIR = r'E:\phdstudy\BiodiversityPHD-2ndpaper\household_expenditure'
CF_DIR  = r'E:\phdstudy\production\input\io_matrices'
REF_DIR = r'E:\phdstudy\reference'
OUT_DIR = r'.\output'
os.makedirs(OUT_DIR, exist_ok=True)

G, S, N = 164, 120, 201
R = G * S
EPSILON = 1e-9
AREA_TO_M2 = 1e7
LU_NAMES = ['Annual_crops', 'Permanent_crops', 'Pasture',
            'Intensive_forestry', 'Extensive_forestry', 'Urban']
YEAR_COL = 33   # 2023
t_start = time.time()

# ── Direct biodiversity intensity q (same method as category 1) ───────────
print('[1/6] Building footprint intensity q ...')
cf = pd.read_csv(os.path.join(CF_DIR, 'GLORIA_Chaudhary_CF_long.csv'))
cf_country = np.zeros((G, 6))
for li, lu in enumerate(LU_NAMES):
    sub = cf[cf['LU_type'] == lu].sort_values('GLORIA_ID')
    cf_country[:, li] = sub['CF_median_PDF_per_m2'].fillna(0).values
cf_sector = np.repeat(cf_country, S, axis=0)
del cf; gc.collect()

mat_sdg = scipy.io.loadmat(os.path.join(IO_DIR, '99 SDG ind 1990-2029.mat'))
lu_2023 = mat_sdg['SDG'][27:33, :, YEAR_COL]
del mat_sdg; gc.collect()
sat = np.einsum('ls,sl->s', lu_2023 * AREA_TO_M2, cf_sector)   # (R,) direct PDF per sector
del cf_sector, lu_2023; gc.collect()

# ── Z, FD -> total output x ─────────────────────────────────────────────
print('[2/6] Loading Z, FD -> x ...')
with h5py.File(os.path.join(IO_DIR, 'UT2023.mat'), 'r') as hf:
    zk = next(k for k in ['UT2023', 'Z', 'Z2023'] + list(hf.keys()) if k in hf)
    Z = hf[zk][:].astype(np.float64)
with h5py.File(os.path.join(IO_DIR, 'FD2023.mat'), 'r') as hf:
    fk = next(k for k in ['FD', 'FD2023'] + list(hf.keys()) if k in hf)
    FD = hf[fk][:].T.astype(np.float64)
x = Z.sum(1) + FD.sum(1)
x_safe = np.where(x > 0, x, 1.0)
del FD; gc.collect()

# ── World-average technology (block-diagonal Lbar reference) ──────────────
print('[3/6] World-average technology + full Leontief L ... (5-20 min)')
Z_world = Z.reshape(G, S, G, S).sum(axis=(0, 2))
x_world = x.reshape(G, S).sum(0)
A_world = Z_world / np.where(x_world > 0, x_world, 1.0)[None, :]
L_world = np.linalg.inv(np.eye(S) * (1 + EPSILON) - A_world)   # 120x120, cheap
del Z_world; gc.collect()

A = np.nan_to_num(Z / x_safe[None, :], nan=0.0, posinf=0.0, neginf=0.0)
del Z; gc.collect()
t0 = time.time()
L = np.linalg.inv(np.eye(R) * (1 + EPSILON) - A)
del A; gc.collect()
print(f'    L done [{time.time() - t0:.0f}s]')

# ── q (real) and qbar (output-weighted global mean, flat across countries) ─
q = np.nan_to_num(sat / x_safe, nan=0.0, posinf=0.0)
q[x <= 0] = 0.0
qbar_sec = sat.reshape(G, S).sum(0) / np.where(x_world > 0, x_world, 1.0)
qbar = np.tile(qbar_sec, G)
del sat; gc.collect()

# ── Four multiplier vectors: c_{L,Q} = q_type @ L_type ─────────────────────
print('[4/6] Four multiplier vectors (real/average L x real/average Q) ...')


def row_times_Lbar(row):
    """row @ Lbar, where Lbar is block-diagonal with each block = L_world."""
    rf = row.reshape(G, S)
    return np.array([rf[r] @ L_world for r in range(G)]).ravel()


cvecs = {
    frozenset({'L', 'Q'}): q @ L,              # real supply chain, real intensity
    frozenset({'L'}):      qbar @ L,           # real supply chain, average intensity
    frozenset({'Q'}):      row_times_Lbar(q),  # average supply chain, real intensity
    frozenset():           row_times_Lbar(qbar),
}
del L; gc.collect()

# ── Target (raw household demand) -> expenditure e, basket shares sigma ────
print('[5/6] Loading Target, building e / sigma inputs ...')
npz = np.load(os.path.join(HOU_DIR, 'Target_2023.npz'))
Target = np.nan_to_num(npz['Target'].astype(np.float64), nan=0.0)   # (R, G*N)
regnam = list(npz['regnam'])
colsum_all = Target.sum(0)                      # total expenditure per group, (G*N,)
country_idx = np.repeat(np.arange(G), N)

pop_raw = pd.read_csv(os.path.join(HOU_DIR, 'Population_by_IncomeGroup.csv'), index_col=0)
mapping = pd.read_csv(os.path.join(HOU_DIR, 'GLORIA_Country_Mapping.csv'))
pop2d = np.zeros((G, N))
for gi in range(G):
    for _, row in mapping[mapping['GLORIA_Index'] == gi + 1].iterrows():
        iso3 = row['Population_ISO3']
        if pd.notna(iso3) and iso3 != 'Not in Population' and iso3 in pop_raw.columns:
            pop2d[gi] += pop_raw[iso3].values
pop = pop2d.ravel()

keep = (pop > 0) & (colsum_all > 0)
e = np.zeros_like(colsum_all)
e[keep] = colsum_all[keep] / pop[keep]
ebar = np.sum(pop[keep] * e[keep]) / pop[keep].sum()

wnorm = np.zeros_like(colsum_all)
wnorm[keep] = pop[keep] / colsum_all[keep]
sigmabar = (Target * wnorm[None, :]).sum(1) / pop[keep].sum()

# proj[lq][a] = c_lq . Target[:,a]  (numerator of sigma(a).c_lq before /colsum)
proj = {lq: (c @ Target) for lq, c in cvecs.items()}
sbar_dot = {lq: float(c @ sigmabar) for lq, c in cvecs.items()}
del Target, npz; gc.collect()

# ── World Bank region (WBR) per country, same reference file as N9/N10 ─────
wbr_map = pd.read_csv(os.path.join(REF_DIR, 'nationlist_categorized_titlecase.csv'))
wbr_by_country = wbr_map.set_index('Country_ID')['WBR'].reindex(np.arange(1, G + 1)).to_numpy()
WBR_NAMES = {'EAP': 'East Asia & Pacific', 'ECA': 'Europe & Central Asia',
             'LAC': 'Latin America & Caribbean', 'MENA': 'Middle East & North Africa',
             'NAM': 'North America', 'SAR': 'South Asia', 'SSA': 'Sub-Saharan Africa'}

idx_all = np.where(keep)[0]
w_all = pop[idx_all]
print(f'[6/6] Ready. {idx_all.size:,} valid (country, bin) groups, '
      f'total population {w_all.sum() / 1e9:.3f} B  [{(time.time() - t_start) / 60:.1f} min]')


[1/6] Building footprint intensity q ...
[2/6] Loading Z, FD -> x ...
[3/6] World-average technology + full Leontief L ... (5-20 min)
    L done [76s]
[4/6] Four multiplier vectors (real/average L x real/average Q) ...
[5/6] Loading Target, building e / sigma inputs ...
[6/6] Ready. 24,886 valid (country, bin) groups, total population 7.481 B  [2.3 min]


## Part 2 — Classic Theil-T between/within-country decomposition, and Method 1 (flat Shapley)

In [6]:
def theil_t(x_arr, w_arr):
    x_arr, w_arr = np.asarray(x_arr, float), np.asarray(w_arr, float)
    m = np.isfinite(x_arr) & np.isfinite(w_arr) & (w_arr > 0) & (x_arr >= 0)
    x_arr, w_arr = x_arr[m], w_arr[m]
    if x_arr.size == 0:
        return np.nan
    w_arr = w_arr / w_arr.sum()
    xbar = np.sum(w_arr * x_arr)
    if xbar <= 0:
        return 0.0
    ratio = x_arr / xbar
    pos = ratio > 0
    return float(np.sum(w_arr[pos] * ratio[pos] * np.log(ratio[pos])))


def theil_between_within(x_arr, w_arr, group):
    """Exact two-level Theil-T decomposition by `group` (e.g. country)."""
    x_arr, w_arr, group = np.asarray(x_arr, float), np.asarray(w_arr, float), np.asarray(group)
    m = np.isfinite(x_arr) & np.isfinite(w_arr) & (w_arr > 0) & (x_arr >= 0)
    x_arr, w_arr, group = x_arr[m], w_arr[m], group[m]
    W = w_arr.sum()
    xbar = np.sum(w_arr * x_arr) / W
    if xbar <= 0:
        return 0.0, 0.0, 0.0
    t_between = t_within = 0.0
    for g in np.unique(group):
        sel = group == g
        wg, xg = w_arr[sel], x_arr[sel]
        Pg = wg.sum()
        xbar_g = np.sum(wg * xg) / Pg
        if xbar_g <= 0:
            continue
        rb = xbar_g / xbar
        t_between += (Pg / W) * rb * np.log(rb)
        rg = xg / xbar_g
        pos = rg > 0
        t_within += (Pg / W) * rb * np.sum((wg[pos] / Pg) * rg[pos] * np.log(rg[pos]))
    return t_between + t_within, t_between, t_within


def x_of_subset(active, idx):
    """BF_per(a) for group subset `idx`, with factors NOT in `active` held at
    their global reference value (ebar / sigmabar)."""
    lq = frozenset(active & {'L', 'Q'})
    E = e[idx] if 'e' in active else ebar
    if 'sigma' in active:
        sc = np.zeros(idx.size)
        ok = colsum_all[idx] > 0
        sc[ok] = proj[lq][idx][ok] / colsum_all[idx][ok]
    else:
        sc = np.full(idx.size, sbar_dot[lq])
    return E * sc


def shapley(factors, I):
    """Exact Shapley value of each factor; contributions sum to I(full set)."""
    from itertools import combinations
    from math import factorial
    s = len(factors)
    cache = {frozenset(c): I(frozenset(c)) for r in range(s + 1) for c in combinations(factors, r)}
    out = {}
    for phi in factors:
        rest = [f for f in factors if f != phi]
        total = 0.0
        for r in range(len(rest) + 1):
            wgt = factorial(r) * factorial(s - r - 1) / factorial(s)
            for c in combinations(rest, r):
                A = frozenset(c)
                total += wgt * (cache[A | {phi}] - cache[A])
        out[phi] = total
    return out, cache[frozenset(factors)]


# ── Classic Theil-T decomposition (no Shapley), by country ─────────────────
idx_pop = np.where(pop > 0)[0]
w_pop = pop[idx_pop]
fp_percap_pop = np.divide(proj[frozenset({'L', 'Q'})][idx_pop], w_pop,
                          out=np.zeros(idx_pop.size, dtype=float), where=w_pop > 0)
T_total, T_between, T_within = theil_between_within(fp_percap_pop, w_pop, country_idx[idx_pop])
fp_percap_all = x_of_subset(frozenset({'e', 'sigma', 'L', 'Q'}), idx_all)
print(f'Theil-T total={T_total:.4f}  between-country={T_between:.4f} '
      f'({T_between / T_total:.1%})  within-country={T_within:.4f} ({T_within / T_total:.1%})')

# ── Method 1: flat 4-factor Shapley of Theil-T ──────────────────────────────
FACTORS = ['e', 'sigma', 'L', 'Q']


def method1_rows(scope, scope_id, scope_name, idx_scope):
    if idx_scope.size < 2:
        return []
    w_scope = pop[idx_scope]
    contrib, total = shapley(FACTORS, lambda A: theil_t(x_of_subset(A, idx_scope), w_scope))
    rows = [{'scope': scope, 'scope_id': scope_id, 'scope_name': scope_name,
             'component': f, 'value': contrib[f],
             'share': contrib[f] / total if total else np.nan,
             'n_groups': int(idx_scope.size), 'population': float(w_scope.sum())}
            for f in FACTORS]
    rows.append({'scope': scope, 'scope_id': scope_id, 'scope_name': scope_name,
                 'component': 'TOTAL', 'value': total, 'share': 1.0,
                 'n_groups': int(idx_scope.size), 'population': float(w_scope.sum())})
    return rows


print('Method 1: global + per-country + per-WBR-region Theil-T Shapley ...')
m1_rows = method1_rows('global', 'global', 'Global', idx_all)

ctry_of_idx = country_idx[idx_all]
for gi in range(G):
    m1_rows += method1_rows('country', gi + 1, regnam[gi], idx_all[ctry_of_idx == gi])

wbr_of_idx = wbr_by_country[ctry_of_idx]
for wbr in WBR_NAMES:
    idx_r = idx_all[wbr_of_idx == wbr]
    m1_rows += method1_rows('wbr', wbr, WBR_NAMES[wbr], idx_r)

df_method1 = pd.DataFrame(m1_rows)
print(df_method1[df_method1['scope'] == 'global'].to_string(index=False))


Theil-T total=0.7433  between-country=0.5027 (67.6%)  within-country=0.2405 (32.4%)
Method 1: global + per-country + per-WBR-region Theil-T Shapley ...
 scope scope_id scope_name component     value     share  n_groups   population
global   global     Global         e  0.848142  1.148564     24886 7.480568e+09
global   global     Global     sigma -0.038277 -0.051835     24886 7.480568e+09
global   global     Global         L -0.110906 -0.150190     24886 7.480568e+09
global   global     Global         Q  0.039478  0.053461     24886 7.480568e+09
global   global     Global     TOTAL  0.738437  1.000000     24886 7.480568e+09


## Part 3 — Method 2: nested Theil-T Shapley (within-country {e, sigma}, between-country {e, sigma, L, Q}), and export

In [7]:
# ── Within-country: 2-factor {e, sigma} Shapley of each country's own Theil-T
print('Method 2: nested within/between Theil-T Shapley ...')
C_e_W = C_sigma_W = T_within_2 = 0.0
c_QL = frozenset({'L', 'Q'})
xbar_all = np.sum(w_all * fp_percap_all) / w_all.sum()

for gi in range(G):
    sel = ctry_of_idx == gi
    if sel.sum() < 2:
        continue
    sub = idx_all[sel]
    wc = pop[sub]
    Pc = wc.sum()
    xbar_c = np.sum(wc * fp_percap_all[sel]) / Pc
    ebar_c = np.sum(wc * e[sub]) / Pc
    sbar_c_dot = np.sum(wc * proj[c_QL][sub] / np.where(colsum_all[sub] > 0, colsum_all[sub], 1.0)) / Pc

    def x_in(active, _sub=sub, _ebar_c=ebar_c, _sbar_c_dot=sbar_c_dot):
        E = e[_sub] if 'e' in active else _ebar_c
        sc = (proj[c_QL][_sub] / colsum_all[_sub]) if 'sigma' in active else _sbar_c_dot
        return E * (sc if np.ndim(sc) else np.full(_sub.size, sc))

    contrib_in, _ = shapley(['e', 'sigma'], lambda A: theil_t(x_in(A), wc))
    weight = (Pc / w_all.sum()) * (xbar_c / xbar_all)
    C_e_W += weight * contrib_in['e']
    C_sigma_W += weight * contrib_in['sigma']
    rc = fp_percap_all[sel] / xbar_c
    T_within_2 += weight * np.sum((wc / Pc) * rc * np.log(rc))

# ── Between-country: aggregate to country means, 4-factor Shapley ──────────
def agg_by_country(v):
    out = np.zeros(G)
    np.add.at(out, ctry_of_idx, v)
    return out


colsum_i, pop_i = agg_by_country(colsum_all[idx_all]), agg_by_country(pop[idx_all])
proj_i = {lq: agg_by_country(proj[lq][idx_all]) for lq in cvecs}
keep_i = pop_i > 0
e_i = np.zeros(G)
e_i[keep_i] = colsum_i[keep_i] / pop_i[keep_i]
idxc, w_i = np.where(keep_i)[0], pop_i[keep_i]
ebar_i = np.sum(w_i * e_i[idxc]) / w_i.sum()
sbar_i = {lq: np.sum(w_i * proj_i[lq][idxc] / np.where(colsum_i[idxc] > 0, colsum_i[idxc], 1.0)) / w_i.sum()
          for lq in cvecs}


def x_between(active):
    lq = frozenset(active & {'L', 'Q'})
    E = e_i[idxc] if 'e' in active else ebar_i
    if 'sigma' in active:
        sc = np.zeros(idxc.size)
        ok = colsum_i[idxc] > 0
        sc[ok] = proj_i[lq][idxc][ok] / colsum_i[idxc][ok]
    else:
        sc = np.full(idxc.size, sbar_i[lq])
    return E * sc


contrib_between, T_between_check = shapley(FACTORS, lambda A: theil_t(x_between(A), w_i))

method2 = {
    'T_between': T_between_check, 'T_within': T_within_2,
    'C_e_within': C_e_W, 'C_sigma_within': C_sigma_W,
    'C_e_between': contrib_between['e'], 'C_sigma_between': contrib_between['sigma'],
    'C_L_between': contrib_between['L'], 'C_Q_between': contrib_between['Q'],
}
print(f"T_between (Shapley-check)={T_between_check:.4f}  (classic decomposition gave {T_between:.4f})")
print(f"Within-country:  C_e={C_e_W:.4f}  C_sigma={C_sigma_W:.4f}")
print(f"Between-country: C_e={contrib_between['e']:.4f}  C_sigma={contrib_between['sigma']:.4f}  "
      f"C_L={contrib_between['L']:.4f}  C_Q={contrib_between['Q']:.4f}")

# ── Export ───────────────────────────────────────────────────────────────
out_xlsx = os.path.join(OUT_DIR, 'theil_shapley_summary_2023.xlsx')
with pd.ExcelWriter(out_xlsx, engine='openpyxl') as w:
    pd.DataFrame({'Metric': ['T_total', 'T_between', 'T_within'],
                  'Value': [T_total, T_between, T_within]}).to_excel(w, sheet_name='Theil_Classic', index=False)
    df_method1[df_method1['scope'] == 'global'].to_excel(w, sheet_name='Method1_Global', index=False)
    df_method1[df_method1['scope'] == 'country'].to_excel(w, sheet_name='Method1_Country', index=False)
    df_method1[df_method1['scope'] == 'wbr'].to_excel(w, sheet_name='Method1_WBR', index=False)
    pd.DataFrame([method2]).to_excel(w, sheet_name='Method2_Nested', index=False)

print(f'\nSaved: {out_xlsx}')


Method 2: nested within/between Theil-T Shapley ...
T_between (Shapley-check)=0.4979  (classic decomposition gave 0.5027)
Within-country:  C_e=0.2966  C_sigma=-0.0560
Between-country: C_e=0.4895  C_sigma=0.0473  C_L=-0.1057  C_Q=0.0668

Saved: .\output\theil_shapley_summary_2023.xlsx
